<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Michi/rulebasedmodel_michi_done.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Rule based model (baseline)**

In [ ]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

Cloning into 'DataScienceCapstoneProject'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 133 (delta 41), reused 20 (delta 20), pack-reused 88 (from 1)
Receiving objects: 100% (133/133), 1.35 MiB | 12.89 MiB/s, done.
Resolving deltas: 100% (74/74), done.


In [ ]:
%cd DataScienceCapstoneProject
!git checkout Michi

/content/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject/DataScienceCapstoneProject
Branch 'Michi' set up to track remote branch 'Michi' from 'origin'.
Switc

In [ ]:
!ls

Data_Exploration.ipynb	     linkedin-cvs-not-annotated.json
data_exploration_v2.ipynb    linkedin-cvs-not-annotated-to-be-predicted
department-v2.csv	     linkedin_experience_annotated.csv
df_profiles_cleansed	     RuleBasedModel-Michi.ipynb
linkedin-cvs-annotated.json  seniority-v2.csv


Data sets required for the rule-based model:


*   df_profiles_cleansed: cleaned data set for testing/evaluation
*   department-v2 & seniority-v2: to define the baseline model



In [ ]:
import pandas as pd
df_profiles_cleansed = pd.read_csv("df_profiles_cleansed")
df_profiles_cleansed.head(15)

,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years
0,Depot4Design GmbH,Prokurist,2019-08,2025-12,ACTIVE,Other,Management,0,6,6.339726
1,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
2,Depot4Design GmbH,Betriebswirtin,2019-07,2025-12,ACTIVE,Other,Professional,0,6,6.424658
3,Depot4Design GmbH,Prokuristin,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
4,Depot4Design GmbH,CFO,2019-07,2025-12,ACTIVE,Other,Management,0,6,6.424658
5,Nagel Car Group,Buchhalterin,2000-05,2019-06,INACTIVE,Other,Professional,0,6,19.095890
6,Computer Solutions,Solutions Architect,2024-03,2025-12,ACTIVE,Information Technology,Professional,1,8,1.753425
7,Computer Solutions,Senior Network Engineer,2019-07,2024-03,INACTIVE,Information Technology,Senior,1,8,4.671233
8,Texas A&M University-Corpus Christi,Manager of Network Services,2017-02,2019-07,INACTIVE,Information Technology,Professional,1,8,2.410959
9,Texas A&M University-Corpus Christi,Infrastructure Administrator II,2015-06,2017-02,INACTIVE,Information Technology,Professional,1,8,1.673973


In [ ]:
df_department = pd.read_csv("department-v2.csv")
df_department.head(15)

,text,label
0,Adjoint directeur communication,Marketing
1,Advisor Strategy and Projects,Project Management
2,Beratung & Projekte,Project Management
3,Beratung & Projektmanagement,Project Management
4,Beratung und Projektmanagement kommunale Partner,Project Management
5,Cadre marketing digital,Marketing
6,Chargé de communication,Marketing
7,Chargé de communication digitale,Marketing
8,Chargé de communication et marketing,Marketing
9,Chargé de Webmarketing SEO/SEA,Marketing


In [ ]:
#top 20 most frequently used words for each department

from collections import Counter
import re
import pandas as pd

def norm(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = (text.replace("ä", "ae")
                .replace("ö", "oe")
                .replace("ü", "ue")
                .replace("ß", "ss"))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return text.split()

def top_tokens_per_label(df_labels, label_col="department", text_col="text", top_n=20):
    labels = list(df_labels[label_col].dropna().unique())
    labels_sorted = sorted(labels)

    top_lists = {}
    for lab in labels_sorted:
        subset = df_labels[df_labels[label_col] == lab].copy()
        subset["title_norm"] = subset[text_col].map(norm)

        counter = Counter()
        for s in subset["title_norm"]:
            STOPWORDS = {"and", "of", "und", "the", "for", "in", "to", "with", "on", "at", "der","des","die"}
            counter.update(t for t in tokenize(s) if t not in STOPWORDS)

        top_lists[lab] = counter.most_common(top_n)

    cols = []
    data = {}
    for lab in labels_sorted:
        cols.extend([(lab, "token"), (lab, "count")])
        data[(lab, "token")] = [t for t, c in top_lists[lab]]
        data[(lab, "count")] = [c for t, c in top_lists[lab]]

    result = pd.DataFrame(data)
    result.columns = pd.MultiIndex.from_tuples(cols)
    result.index = range(1, top_n + 1)
    result.index.name = "rank"
    return result


top20_department_table = top_tokens_per_label(
    df_department,
    label_col="label",
    text_col="text",
    top_n=20
)

display(top20_department_table)

Administrative       Business Development            Consulting  \
                   token count                token count           token   
rank                                                                        
1            assistentin    37             business   621      consultant   
2              assistenz    34          development   296          senior   
3      geschaeftsleitung    18              manager   190         berater   
4     geschaeftsfuehrung    16                 head    78             sap   
5              assistent    10                  crm    59              it   
6                     gf     5             director    59     recruitment   
7                 office     5               senior    51      management   
8              assistant     5           management    38         inhouse   
9            sekretaerin     4              analyst    37        dynamics   
10                    gl     3                   it    37       microsoft   
11                   ceo     3               leiter    28         digital   
12            management     3              digital    27         manager   
13     projektmanagement     2              process    27         trainer   
14                   von     2                  new    27             erp   
15              vorstand     2               global    25        services   
16           sekretariat     2         intelligence    24             nav   
17                  fuer     2              account    22           coach   
18                   kfm     2        international    21              bi   
19               leitung     2           consultant    20       beraterin   
20            controller     2                 unit    20  projektmanager   

           Customer Support       Human Resources        ...       Marketing  \
     count            token count           token count  ...           token   
rank                                                     ...                   
1      140          support    29              hr    23  ...       marketing   
2       41               it    15         manager     8  ...         manager   
3       22          manager     7           human     7  ...           sales   
4       22         customer     6       resources     5  ...            head   
5       19        technical     5     assistentin     3  ...   communication   
6       10        supporter     4        director     3  ...  communications   
7       10              1st     2            head     3  ...        director   
8        9          systems     2      management     2  ...          leiter   
9        8   administration     2          office     2  ...        vertrieb   
10       7           global     2       assistant     2  ...   kommunikation   
11       7             head     2      recruiting     2  ...           messe   
12       6          service     2       abteilung     1  ...          senior   
13       5           leiter     2   buerokauffrau     1  ...         digital   
14       5          account     2       assistenz     1  ...         leitung   
15       5       management     2              gl     1  ...      specialist   
16       5             line     1        heavenhr     1  ...              pr   
17       4          backend     1         adviser     1  ...          global   
18       4              edv     1              it     1  ...        business   
19       4              erp     1       ressource     1  ...           event   
20       4        microsoft     1   administrator     1  ...             crm   

                 Other       Project Management                 Purchasing  \
     count       token count              token count                token   
rank                                                                         
1     3745  operations    42            project    66              einkauf   
2      860     manager    12            manager    42           purchasing   
3      

In [ ]:
df_seniority = pd.read_csv("seniority-v2.csv")
df_seniority.head(15)

,text,label
0,Analyst,Junior
1,Analyste financier,Junior
2,Anwendungstechnischer Mitarbeiter,Junior
3,Application Engineer,Senior
4,Applications Engineer,Senior
5,Architecte SI - Chef de projet Applicatif,Lead
6,Associate,Junior
7,Associate - Research,Junior
8,Associate Partner,Junior
9,Associate Recruiter,Junior


In [ ]:
#top 20 most frequently used words for each seniority

def norm(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = (text.replace("ä", "ae")
                .replace("ö", "oe")
                .replace("ü", "ue")
                .replace("ß", "ss"))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return text.split()

def top_tokens_per_label(df_labels, label_col="seniority", text_col="text", top_n=20):
    labels = list(df_labels[label_col].dropna().unique())
    labels_sorted = sorted(labels)

    top_lists = {}
    for lab in labels_sorted:
        subset = df_labels[df_labels[label_col] == lab].copy()
        subset["title_norm"] = subset[text_col].map(norm)

        counter = Counter()
        for s in subset["title_norm"]:
            STOPWORDS = {"and", "of", "und", "the", "for", "in", "to", "with", "on", "at"}
            counter.update(t for t in tokenize(s) if t not in STOPWORDS)

        top_lists[lab] = counter.most_common(top_n)

    cols = []
    data = {}
    for lab in labels_sorted:
        cols.extend([(lab, "token"), (lab, "count")])
        data[(lab, "token")] = [t for t, c in top_lists[lab]]
        data[(lab, "count")] = [c for t, c in top_lists[lab]]

    result = pd.DataFrame(data)
    result.columns = pd.MultiIndex.from_tuples(cols)
    result.index = range(1, top_n + 1)
    result.index.name = "rank"
    return result


top20_table = top_tokens_per_label(df_seniority, label_col="label", text_col="text", top_n=20)
display(top20_table)

Director                           Junior        \
               token count                      token count   
rank                                                          
1           director   966                  marketing   173   
2              sales   453                     junior    88   
3          marketing   265                    analyst    68   
4           business   101                 referentin    55   
5        development    80                assistentin    55   
6           managing    73                   business    45   
7             global    70              mitarbeiterin    39   
8             senior    47                   referent    38   
9             europe    42                        crm    37   
10              dach    40                mitarbeiter    36   
11              emea    39                      sales    35   
12     international    38                    manager    33   
13        management    36                   vertrieb    31   
14           digital    35              kommunikation    23   
15           germany    30                         it    20   
16    communications    29                  associate    16   
17             group    28                  assistent    16   
18                it    28  unternehmenskommunikation    14   
19                 d    26                       fuer    14   
20                ch    24                development    12   

                   Lead                Management               Senior        
                  token count               token count          token count  
rank                                                                          
1                  head   999   geschaeftsfuehrer   165        manager  2426  
2             marketing   951               sales   127      marketing  1089  
3                leiter   781           president   124          sales   859  
4                 sales   586                vice   119         senior   564  
5              vertrieb   489           marketing   117     management   401  
6               leitung   400                 ceo    98       business   366  
7                    it   223               chief    80            crm   355  
8                   crm   197  geschaeftsfuehrung    79        account   289  
9       vertriebsleiter   193             officer    73     consultant   266  
10             business   190             founder    72    development   246  
11             leiterin   142                  vp    70        project   177  
12           teamleiter   132            vertrieb    52             it   177  
13          development   127                 der    49    responsable   157  
14    geschaeftsleitung   107            business    46            key   155  
15                  der   104               owner    42      assistant   137  
16        kommunikation   101                  co    39        digital   136  
17        projektleiter    88         development    35      managerin   135  
18            prokurist    86             product    33           head   133  
19              digital    82      gesellschafter    27  communication   130  
20       bereichsleiter    76              global    26         global   122

Rule based model: defining current job -> Rule: ACTIVE jobs

In [ ]:
#For repeated executions when multiple 'row_idx' columns already exist
df_profiles_cleansed = df_profiles_cleansed.loc[:, ~df_profiles_cleansed.columns.duplicated()]
if 'row_idx' in df_profiles_cleansed.columns:
    df_profiles_cleansed = df_profiles_cleansed.drop(columns=['row_idx'])
df_profiles_cleansed = df_profiles_cleansed.reset_index(drop=True)
df_profiles_cleansed['row_idx'] = df_profiles_cleansed.index

#parsing date -> Date values are converted to datetime so that ‘latest job’ can be sorted correctly
df_profiles_cleansed["startDate"] = pd.to_datetime(df_profiles_cleansed["startDate"], errors="coerce")
df_profiles_cleansed["endDate"] = pd.to_datetime(df_profiles_cleansed["endDate"], errors="coerce")

#current job -> only one active job per person
  #Rule: If ACTIVE exists → take the ACTIVE job with the latest startDate (Fallback: latest endDate → fallback: row_idx);
  #Otherwise → take the INACTIVE job with the latest endDate (Fallback: latest startDate → fallback: row_idx)

def select_current_job(group):
    g = group.copy()

    active = g[g["status"] == "ACTIVE"]
    if len(active) > 0:
        return (
            active
            .sort_values(
                by=["startDate", "endDate", "row_idx"],
                ascending=[False, False, False]
            )
            .iloc[0]
        )

    inactive = g[g["status"] != "ACTIVE"]
    return (
        inactive
        .sort_values(
            by=["endDate", "startDate", "row_idx"],
            ascending=[False, False, False]
        )
        .iloc[0]
    )

current_df = (
    df_profiles_cleansed
    .groupby("person_id", group_keys=False)
    .apply(select_current_job)
    .reset_index(drop=True)
)

current_df.head()

/tmp/ipython-input-1226484754.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_current_job)


,organization,position,startDate,endDate,status,department,seniority,person_id,job_count,job_duration_years,row_idx
0,Depot4Design GmbH,Prokurist,2019-08-01,2025-12-01,ACTIVE,Other,Management,0,6,6.339726,0
1,Computer Solutions,Solutions Architect,2024-03-01,2025-12-01,ACTIVE,Information Technology,Professional,1,8,1.753425,6
2,Udo Weber,Medizintechnik Beratung,2025-01-01,2025-12-01,ACTIVE,Consulting,Professional,2,3,0.915068,14
3,Grupo Viajes Kontiki.,Director expansión de negocio.,2024-09-01,2025-12-01,ACTIVE,Business Development,Director,3,7,1.249315,17
4,Himmelstalunds Utbildningscentrum,"APL-ansvarig, samordning",2019-08-01,2025-12-01,ACTIVE,Administrative,Lead,4,8,6.339726,24


In [ ]:
print(df_profiles_cleansed.columns)
print(df_seniority.columns)
print(df_department.columns)

Index(['organization', 'position', 'startDate', 'endDate', 'status',
       'department', 'seniority', 'person_id', 'job_count',
       'job_duration_years', 'row_idx'],
      dtype='object')
Index(['text', 'label'], dtype='object')
Index(['text', 'label'], dtype='object')


In [ ]:
#text normalization -> normalise multiple spaces and punctuation marks/separators to space
import re

def norm(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
#dictionaries (Fallback)
sen_map = dict(zip(df_seniority["text"].map(norm), df_seniority["label"]))
dep_map = dict(zip(df_department["text"].map(norm), df_department["label"]))

print("sen_map:", len(sen_map), "entries")
print("dep_map:", len(dep_map), "entries")

sen_map: 9063 entries
dep_map: 9689 entries


**Rules for seniority und departments**




In [ ]:
#seniority rules
def has_word(t: str, token: str) -> bool:
    return re.search(rf"\b{re.escape(token)}\b", t) is not None

def predict_seniority(title):
    t = norm(title)

    #hard rules
    if any(k in t for k in [
        "ceo","cfo","cto","cmo","chief","vice","president","owner","gesellschafter","founder","cofounder","co-founder","prokurist","prokuristin","unternehmensinhaber","gesch"]):
        return "Management", "keyword_mgmt"

    if any(k in t for k in [
        "director", "global", "international", "dach"]):
        return "Director", "keyword_director"

    if any(k in t for k in [
        "lead","leitung","leiter","head"]):
        return "Lead", "keyword_lead"

    if any(k in t for k in [
        "senior"]or has_word(t, "sr")):
        return "Senior", "keyword_senior"

    if any(k in t for k in [
        "intern","junior","trainee","student","werkstudent","referent","assistent","auszubild"]):
        return "Junior", "keyword_junior"

    #fallback exact match in label list
    if t in sen_map:
        return sen_map[t], "list_exact"

    return "Professional", "default"

In [ ]:
#department rules
def has_word(t: str, token: str) -> bool:
    return re.search(rf"\b{re.escape(token)}\b", t) is not None

def predict_department(title, organization=None):
    t = norm(title)

    #hard rules
    #Administrative
    if any(k in t for k in [
        "assistentin","assistenz","assistent","office","assistant","sekret"]):
        return "Administrative", "keyword_administrative"

    #Business Development
    if any(k in t for k in [
        "business","development"] or has_word(t, "bd")):
        return "Business Development", "keyword_bd"

    #Consulting
    if any(k in t for k in [
        "consultant","berater","sap","dynamics","erp"]):
        return "Consulting", "keyword_consulting"

    #Customer Support
    if any(k in t for k in [
        "support","customer","supporter"]):
        return "Customer Support", "keyword_customer_support"

    #Information Technology
    if any(k in t for k in [
        "software","developer","engineer","architect","devops","cloud","data scientist","data engineer",
        "network","systems administrator","administrator"] or has_word(t, "it")):
        return "Information Technology", "keyword_it"

    #Human Resources
    if any(k in t for k in [
        "human","resources","ressource"]):
        return "Human Resources", "keyword_hr"

    #Marketing
    if any(k in t for k in [
        "marketing","communication","communications","kommunikation","messe","event"]):
        return "Marketing", "keyword_marketing"

    #Project Management
    if any(k in t for k in [
        "project","projektleiter","projektmanager","projektmanagement","projekt",
        "projektleitung","projects"]):
        return "Project Management", "keyword_project_mgmt"

    #Purchasing
    if any(k in t for k in [
        "einkauf","purchas","eink"]):
        return "Purchasing", "keyword_purchasing"

    #Sales
    if any(k in t for k in [
        "sales","vertrieb","vertriebsleiter","salesforce"]):
        return "Sales", "keyword_sales"

    #fallback exact match in label list
    if t in dep_map:
        return dep_map[t], "list_exact"

    return "Other", "default"

**Evaluation**

In [ ]:
#predictions of current_df
from sklearn.metrics import classification_report, accuracy_score

rows = []
for _, r in current_df.iterrows():
    dep_pred, dep_reason = predict_department(r["position"], r.get("organization"))
    sen_pred, sen_reason = predict_seniority(r["position"])

    rows.append({
        "person_id": r["person_id"],
        "title": r["position"],
        "org": r["organization"],
        "true_department": r["department"],
        "pred_department": dep_pred,
        "dep_reason": dep_reason,
        "true_seniority": r["seniority"],
        "pred_seniority": sen_pred,
        "sen_reason": sen_reason,
    })

df_pred = pd.DataFrame(rows)

#accuracy
dept_acc = (df_pred["pred_department"] == df_pred["true_department"]).mean()
sen_acc  = (df_pred["pred_seniority"] == df_pred["true_seniority"]).mean()

print(f"Department accuracy: {dept_acc:.2%}")
print(f"Seniority accuracy:  {sen_acc:.2%}")


print("\nDepartment classification report:")
print(classification_report(
    df_pred["true_department"],
    df_pred["pred_department"],
    digits=3,
    zero_division=0
))

print("\nSeniority classification report:")
print(classification_report(
    df_pred["true_seniority"],
    df_pred["pred_seniority"],
    digits=3,
    zero_division=0
))

Department accuracy: 71.17%
Seniority accuracy:  71.66%

Department classification report:
                        precision    recall  f1-score   support

        Administrative      0.240     0.316     0.273        19
  Business Development      0.333     0.353     0.343        17
            Consulting      0.793     0.622     0.697        37
      Customer Support      0.750     0.429     0.545         7
       Human Resources      0.833     0.263     0.400        19
Information Technology      0.833     0.448     0.583        67
             Marketing      0.857     0.500     0.632        24
                 Other      0.705     0.908     0.794       327
    Project Management      0.857     0.545     0.667        33
            Purchasing      1.000     0.571     0.727        14
                 Sales      0.960     0.558     0.706        43

              accuracy                          0.712       607
             macro avg      0.742     0.501     0.579       607
          w

The rule-based model performs quite well overall, despite the purely rule-based approach and multilingual job titles. Low values are likely due in part to class imbalance.

Department: High performance values are achieved in the department, especially for sales, marketing and consulting, as clearly recognisable keywords are marked here (e.g. sales, consulting, etc.) and are also included in most job titles. IT has relatively good precision, but much poorer recall, which suggests that IT roles are often recognised, but some are not captured due to semantic diversity.

Seniority: The predictions for professional and management achieve the highest performance values here. Management also benefits from clearly recognisable keywords; professional from the default classification. As with department purchasing, Lead shows a precision of 1 but a lower recall. This means that the positions are clearly recognised, but some of the actual cases are overlooked, probably due to overly narrowly defined rules. Director is the weakest class, probably due to content overlap with management roles and/or inconsistent use of titles.


In [ ]:
#error analysis

dept_errors = df_pred[df_pred["pred_department"] != df_pred["true_department"]]
sen_errors  = df_pred[df_pred["pred_seniority"] != df_pred["true_seniority"]]

print("Department errors:", len(dept_errors))
print("Seniority errors:", len(sen_errors))

#department errors
dept_errors[["title","true_department","pred_department","dep_reason"]].head(15)

Department errors: 175
Seniority errors: 172


,title,true_department,pred_department,dep_reason
2,Medizintechnik Beratung,Consulting,Other,default
3,Director expansión de negocio.,Business Development,Other,default
4,"APL-ansvarig, samordning",Administrative,Other,default
5,Kaufmännischer Leiter,Sales,Other,default
10,Applikation Analyst Senior,Information Technology,Other,default
17,Abfallberaterin,Other,Consulting,keyword_consulting
21,"Geschäftsführer, CMO",Marketing,Other,default
24,Physician Assistant,Other,Administrative,keyword_administrative
26,Consultor de vendas,Sales,Other,default
36,Head of HR/CHRO,Human Resources,Other,default


In [ ]:
#top 10 department errors

dept_errors_examples = (
    dept_errors
    .groupby(["true_department", "pred_department"])
    .agg(
        count=("title", "size"),
        examples=("title", lambda x: x.head(3).tolist())
    )
    .reset_index()
    .sort_values("count", ascending=False)
    .head(10)
)

dept_errors_examples

,true_department,pred_department,count,examples
13,Information Technology,Other,30,"[Applikation Analyst Senior, Technical Team Le..."
30,Sales,Other,15,"[Kaufmännischer Leiter, Consultor de vendas, A..."
17,Other,Administrative,15,"[Physician Assistant, Chief Scientific Officer..."
9,Human Resources,Other,13,"[Head of HR/CHRO, Trainer/docent, Personalleiter]"
25,Project Management,Other,13,"[Product Manager, Sub-Stream Lead Simplify | T..."
0,Administrative,Other,13,"[APL-ansvarig, samordning, Büroleiter | Presse..."
1,Business Development,Other,11,"[Director expansión de negocio., Strategic Eng..."
16,Marketing,Other,10,"[Geschäftsführer, CMO, Strategic Design Direct..."
5,Consulting,Other,9,"[Medizintechnik Beratung, Practice Leader, Dat..."
26,Purchasing,Other,6,"[Strategisch inkoper, OÖ Koordinator für das T..."


In [ ]:
#Department errors -> per misclassification (True → Pred) the top 10 most common job titles

def top_titles_per_error(
    df_errors,
    true_col,
    pred_col,
    title_col="title",
    top_n_titles=10,
    top_n_errors=10
):

    #most common incorrect combinations
    top_errors = (
        df_errors
        .groupby([true_col, pred_col])
        .size()
        .reset_index(name="error_count")
        .sort_values("error_count", ascending=False)
        .head(top_n_errors)
    )

    tables = {}

    for _, row in top_errors.iterrows():
        t_true = row[true_col]
        t_pred = row[pred_col]

        subset = df_errors[
            (df_errors[true_col] == t_true) &
            (df_errors[pred_col] == t_pred)
        ]

        top_titles = (
            subset
            .groupby(title_col)
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
            .head(top_n_titles)
        )

        tables[(t_true, t_pred)] = top_titles

    return tables

department_error_tables = top_titles_per_error(
    dept_errors,
    true_col="true_department",
    pred_col="pred_department",
    title_col="title",
    top_n_titles=10,
    top_n_errors=10
)

from IPython.display import display

for (true_d, pred_d), table in department_error_tables.items():
    print(f"\nTRUE = {true_d}  →  PRED = {pred_d}")
    display(table)


TRUE = Information Technology  →  PRED = Other


,title,count
3,CTO,3
0,Applikation Analyst Senior,1
1,Betriebsleiter API-Produktion,1
2,Bilgisayar mühendisi,1
4,Cyber Security Analyst - Senior,1
5,Data Analyst,1
6,DevSecOps CoE Lead,1
7,Director Code Nomads Amsterdam,1
8,Head Of RIV Academy - Blockchain,1
9,Head of ICT,1



TRUE = Sales  →  PRED = Other


,title,count
0,Account manager / Sælger Helly Hansen Workwear...,1
1,Außendienstspezialist,1
2,Co-Head of Bid Management CE,1
3,Consultor de vendas,1
4,Kaufmännischer Leiter,1
5,"Key Account Director Europe, Middle East & Africa",1
6,Kundansvarig,1
7,Manager Regional Key Accounts Americas,1
8,Marketplace Manager / Online Shop Manager,1
9,Owner - CSO,1



TRUE = Other  →  PRED = Administrative


...

,title,count
0,Conversion Rate Optimization Manager,1
1,Director Commercial Operations,1
2,"Geschäftsführer, CMO",1
3,Grafik Designer,1
4,Grafikdesigner (Freelance),1
5,Grafiker,1
6,Head of Content & Workflows,1
7,Leiterin E-Commerce,1
8,Referent Digitale Schiene,1
9,Strategic Design Director,1



TRUE = Consulting  →  PRED = Other


,title,count
0,Deloitte Digital,1
1,Executive Advisor,1
2,Founder and Senior Partner,1
3,Lehrbeauftragter,1
4,Medizintechnik Beratung,1
5,"Practice Leader, Data Valorization Strategy",1
6,Research Associate,1
7,Strategic Planning,1
8,Technical Transfer Leader,1



TRUE = Purchasing  →  PRED = Other


,title,count
0,Buyer,1
1,"Buying, Porecurement & Allocation",1
2,Head of Procurement,1
3,OÖ Koordinator für das Team der Bereiche Besch...,1
4,Sourcing Specialist,1
5,Strategisch inkoper,1


In [ ]:
#seniority errors

sen_errors[["title","true_seniority","pred_seniority","sen_reason"]].head(15)

,title,true_seniority,pred_seniority,sen_reason
3,Director expansión de negocio.,Director,Management,keyword_mgmt
4,"APL-ansvarig, samordning",Lead,Professional,default
6,Lab-Supervisor,Lead,Professional,default
9,Meister,Lead,Professional,default
15,Purchasing Manager,Professional,Senior,list_exact
23,Principal RF Design Engineer,Senior,Professional,default
30,Qualifizierter Kreditsachbearbeiter im risikor...,Professional,Management,keyword_mgmt
31,Director,Director,Management,keyword_mgmt
38,Office Associate,Junior,Professional,default
40,Manager,Professional,Senior,list_exact


In [ ]:
#top 10 seniority errors
sen_errors_examples = (
    sen_errors
    .groupby(["true_seniority", "pred_seniority"])
    .agg(
        count=("title", "size"),
        examples=("title", lambda x: x.head(3).tolist())
    )
    .reset_index()
    .sort_values("count", ascending=False)
    .head(10)
)

sen_errors_examples

,true_seniority,pred_seniority,count,examples
7,Lead,Professional,34,"[APL-ansvarig, samordning, Lab-Supervisor, Mei..."
10,Management,Professional,32,"[Alleinvorstand Roto Frank Holding AG, Member ..."
0,Director,Management,27,"[Director expansión de negocio., Director, Dir..."
15,Professional,Senior,21,"[Purchasing Manager, Manager, Product Manager]"
6,Lead,Management,10,[Technical Customer Service Manager - Flexible...
17,Senior,Professional,8,"[Principal RF Design Engineer, Sr. Application..."
2,Junior,Professional,8,"[Office Associate, Duales Studium Diplom Finan..."
14,Professional,Management,8,[Qualifizierter Kreditsachbearbeiter im risiko...
13,Professional,Junior,5,"[Data Analyst, Business Analyst, Bestuursassis..."
11,Management,Senior,4,"[General Manager, General Manager, General Man..."


In [ ]:
# per misclassification (True → Pred) the top 10 most common job titles
seniority_error_tables = top_titles_per_error(
    sen_errors,
    true_col="true_seniority",
    pred_col="pred_seniority",
    title_col="title",
    top_n_titles=10,
    top_n_errors=10
)

from IPython.display import display

for (true_s, pred_s), table in seniority_error_tables.items():
    print(f"\nTRUE = {true_s}  →  PRED = {pred_s}")
    display(table)


TRUE = Lead  →  PRED = Professional


,title,count
0,"APL-ansvarig, samordning",1
1,Accounting Manager,1
2,Bereichs PDL,1
3,Communications Coordinator,1
4,Country Manager Czech Republic and Slovakia,1
5,Delegierter Verwalter,1
6,District Sales manager,1
7,Facility Coordinator,1
8,Hoofd Bouwbureau,1
9,Interims Technical Manager,1



TRUE = Management  →  PRED = Professional


,title,count
17,Partner,5
0,Algemeen directeur,1
2,Arbeitgeber,1
1,Alleinvorstand Roto Frank Holding AG,1
4,Eigenaar,1
5,Executive Advisor,1
6,Founding Partner,1
7,General Manager:in,1
8,Godsejer,1
9,Inhaber,1



TRUE = Director  →  PRED = Management


...

,title,count
0,Commercial Service Manager,1
1,Freelance Marketing Consulting / Client Servic...,1
2,Legal Services,1
3,Postdoctoral Research Fellow,1
4,Postdoctoral Researcher and Lecturer,1
5,Product and Service Manager Qualitymanagement;...,1
6,Qualifizierter Kreditsachbearbeiter im risikor...,1
7,Warranty Service Specialist,1



TRUE = Professional  →  PRED = Junior


,title,count
0,Bestuursassistent,1
1,Business Analyst,1
2,Data Analyst,1
3,Referent Digitale Schiene,1
4,Research Analyst,1



TRUE = Management  →  PRED = Senior


,title,count
0,General Manager,3
1,Managing Partner,1


The rule-based baseline achieves in the first implementation with small adjustments 63% accuracy for seniority prediction and 56% for department prediction.


The following improvements/enhancements have been added retrospectively:

*   ‘it’ and ‘bd’ have been replaced as word boundaries; fallbacks for missing start and end dates have been added

---


    Accuracy *new*: department 63% (+5%), seniority 61% (-2%)




*   Adjustment of keywords for seniority (e.g. remove ‘manager’ from senior, etc.)

---



    Accuracy *new*: department 63% (+0%), seniority 71% (+10%)




*   Adjustment of keywords for department (e.g. remove ‘hr’ from human resources, etc.)

---



    Accuracy *new*: department 71% (+7%), seniority 71% (+0%)





**Pipeline:**

In [ ]:
def pipeline(job_title, organization=None):
    dep, dep_reason = predict_department(job_title, organization)
    sen, sen_reason = predict_seniority(job_title)
    return {
        "department": dep,
        "seniority": sen,
        "dep_reason": dep_reason,
        "sen_reason": sen_reason
    }

In [ ]:
pipeline("CMO")

{'department': 'Other',
 'seniority': 'Management',
 'dep_reason': 'default',
 'sen_reason': 'keyword_mgmt'}

In [ ]:
tests = ["CMO", "CFO", "Senior Network Engineer", "Marketing Intern", "Human Resources Generalist", "Junior Consultant"]
for t in tests:
    print(t, "->", pipeline(t))

CMO -> {'department': 'Other', 'seniority': 'Management', 'dep_reason': 'default', 'sen_reason': 'keyword_mgmt'}
CFO -> {'department': 'Other', 'seniority': 'Management', 'dep_reason': 'default', 'sen_reason': 'keyword_mgmt'}
Senior Network Engineer -> {'department': 'Information Technology', 'seniority': 'Senior', 'dep_reason': 'keyword_it', 'sen_reason': 'keyword_senior'}
Marketing Intern -> {'department': 'Marketing', 'seniority': 'Junior', 'dep_reason': 'keyword_marketing', 'sen_reason': 'keyword_junior'}
Human Resources Generalist -> {'department': 'Human Resources', 'seniority': 'Professional', 'dep_reason': 'keyword_hr', 'sen_reason': 'default'}
Junior Consultant -> {'department': 'Consulting', 'seniority': 'Junior', 'dep_reason': 'keyword_consulting', 'sen_reason': 'keyword_junior'}
